missing values

In [3]:
import pandas as pd
import os
from pathlib import Path
p = Path.cwd()
while not (p / 'data').exists() and p != p.parent:
    p = p.parent
os.chdir(p)
print('рабочая папка:', os.getcwd())

рабочая папка: c:\Users\sea\Desktop\---\housing-price-econometrics


In [4]:
#загружаем данные
df = pd.read_csv('data/raw/AmesHousing.csv', encoding='latin1')

| Column | Missing | Type | Action |
|---|---|---|---|
| Pool QC | 2917 | Нет объекта | Заполнить "None" |
| Misc Feature | 2824 | Нет объекта | Заполнить "None" |
| Alley | 2732 | Нет объекта | Заполнить "None" |
| Fence | 2358 | Нет объекта | Заполнить "None" |
| Mas Vnr Type | 1775 | Нет объекта | Заполнить "None" |
| Fireplace Qu| 1422 | Нет объекта | Заполнить "None" |
| Lot Frontage | 490 | Неизвестно | Заполнить медианой/ср |
| Garage Qual | 159 | Нет объекта | Заполнить "No Garage" or None|
| Garage Cond | 159 | Нет объекта | Заполнить "No Garage" oe None|
| Garage Yr Blt | 159 | Нет объекта | Заполнить 0 |
| Garage Finish | 159 | Нет объекта | Заполнить "None" |
| Garage Type | 157 | Нет объекта | Заполнить "No Garage" or None |
| Bsmt Exposure  | 83 | Нет объекта| Заполнить "None" |
| BsmtFin Type 2  | 81 | Нет объекта | Заполнить "None" |
| Bsmt Cond | 80 | Нет объекта  | Заполнить "None" |
| Bsmt Qual| 80 | Нет объекта | Заполнить "None" |
| BsmtFin Type 1 | 80 | Нет объекта | ЗЗаполнить "None" |
| Mas Vnr Area| 23 | Нет объекта | Заполнить 0 |
| Bsmt Full Bath | 2 | Нет объекта  | Заполнить 0|
| Bsmt Half Bath  | 2 | Нет объекта  | Заполнить 0 |
| BsmtFin SF 1 | 1 | Нет объекта  | Заполнить 0 |
| BsmtFin SF 2 | 1 | Нет объекта  | Заполнить 0 |
| Electrical | 1 | Неизвестно | Заполнить модой |
| Total Bsmt SF | 1 | Нет объекта  | Заполнить 0 |
| Bsmt Unf SF | 1 | Нет объекта  | Заполнить 0 |
| Garage Area | 1 | Нет объекта | Заполнить 0 |
| Garage Cars | 1 | Нет объекта | Заполнить 0 |


Выше в таблице представлен анализ наших пропусков. Данные делятся на категориальные - это те которые надо заполнить "None" и числовые - которые заполнить 0. Но также у нас в данных есть два исключюения: Electrical - "неизвестно" потому что разные типы электричества (мы считаем, что во всех домах есть эл), Lot Frontage -"неизветсно", потому что фасад есть у всех домов, в этих данных просто не заипсан.

In [5]:
cat_cols = ['Pool QC', 'Misc Feature', 'Alley', 'Fence', 'Mas Vnr Type',
            'Fireplace Qu', 'Garage Qual', 'Garage Cond', 'Garage Finish',
            'Garage Type', 'Bsmt Exposure', 'BsmtFin Type 2', 'Bsmt Cond',
            'Bsmt Qual', 'BsmtFin Type 1']
for col in cat_cols:
    df[col] = df[col].fillna('None')

num_cols = ['Garage Yr Blt', 'Mas Vnr Area', 'Bsmt Full Bath',
            'Bsmt Half Bath', 'BsmtFin SF 1', 'BsmtFin SF 2',
            'Total Bsmt SF', 'Bsmt Unf SF', 'Garage Area', 'Garage Cars']
for col in num_cols:
    df[col] = df[col].fillna(0)

df['Lot Frontage'] = df.groupby('Neighborhood')['Lot Frontage'].transform(
    lambda x: x.fillna(x.median())
)

df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])

c:\Users\sea\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\sea\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [7]:
print(df.isnull().sum().sum())  

3


Видно, что у нас есть еще три пропуска, выводим на экран:

In [8]:
print(df.isnull().sum()[df.isnull().sum() > 0])

Lot Frontage    3
dtype: int64


Мы заполнили Lot Frontage медианой по Neighborhood. Т.е. мы пыталисб посчитать медиану, но в группе все значения - NaN. Медиана пустой серии - это NaN. Следовательно у нас есть 3 дома, где в каком-то районе или нескольких, все зн Lot Frontage пропущены. 
Найдем районы, где есть  такие пропуски:

In [9]:
neighborhoods_with_na = df[df['Lot Frontage'].isnull()]['Neighborhood'].unique()
print(neighborhoods_with_na)

['GrnHill' 'Landmrk']


In [10]:
df['Lot Frontage'] = df['Lot Frontage'].fillna(df['Lot Frontage'].median())

In [11]:
print(df.isnull().sum().sum())  

0


Сохраняем данные:

In [12]:
import os
os.makedirs('data/interim', exist_ok=True)
df.to_csv('data/interim/ames_cleaned.csv', index=False)